In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, max as _max, datediff, current_date

spark = SparkSession.builder.appName("InactiveCustomers").getOrCreate()

# Sample Data

data = [
    ("C1", "2024-01-01"),
    ("C1", "2024-02-01"),
    ("C2", "2024-01-10"),
    ("C3", "2023-12-01")
]

columns = ["customer_id", "txn_date"]

df = spark.createDataFrame(data, columns)

# Step 1: Get last transaction date per customer

last_txn_df = df.groupBy("customer_id") \
                .agg(_max("txn_date").alias("last_txn_date"))

# Step 2: Find inactive customers (>30 days)

result_df = last_txn_df.withColumn(
    "days_inactive",
    datediff(current_date(), col("last_txn_date"))
).filter(col("days_inactive") > 30)

result_df.show()